# 01 - Synthetic end-to-end

Runs the whole preprocessing pipeline on a fixture whose correct answer is known by construction,
so every number below can be checked rather than admired.

**What the fixture contains** (`panelprep/io/synthetic.py`): cells on a 2D sheet, expression drawn
from `Poisson(exp(loadings @ spatial_program))`, and four candidate datasets in known order of
similarity to the query:

| candidate | relationship to the query |
|---|---|
| `sibling` | same loadings, same geometry, new cells - **expected best** |
| `same_program_diff_mixture` | same gene-gene structure, different cell-type proportions |
| `shuffled_structure` | each gene permuted **after** normalisation: marginals identical, co-expression destroyed |
| `unrelated` | independent loadings and geometry - **expected last** |

`shuffled_structure` is the discriminating case: a distribution-based score cannot tell it from a
genuine sibling, and a structure-based score must.

In [ ]:
import os, sys, warnings
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))  # import panelprep from the repo root
os.chdir(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else None
warnings.filterwarnings('ignore', category=FutureWarning)
import numpy as np, pandas as pd
from panelprep import Config
print('panelprep ready; cwd =', os.getcwd())

In [ ]:
from panelprep.io.synthetic import make_synthetic_case, EXPECTED_ORDER

case = make_synthetic_case(seed=0)
print('panel (the measured gene subset):', case.query.panel)
print('query:', case.query.expression.shape, 'coords:', case.query.coords.shape, '|', case.query.units)
for name, c in case.candidates.items():
    print(f'  {name:28s} {c.category:14s} {c.expression.shape}  coords={c.has_coords}')

## Stage 1a - descriptor to candidate datasets

Text match over catalogue metadata. This says which datasets plausibly describe the same biology; it is deliberately separate from whether the measured genes actually agree.

In [ ]:
from panelprep.match import match_descriptor

entries = [{'name': n, 'category': c.category, 'stage': c.meta.get('stage'),
            'organism': c.meta.get('organism'), 'modality': c.meta.get('modality'),
            'description': c.meta.get('description')} for n, c in case.candidates.items()]
match_table = match_descriptor(case.query.descriptor, entries, method='tfidf')
match_table

## Stage 1b - agreement scores

Every score is a registry entry. `Config.scores` selects which run; nothing else needs editing.

In [ ]:
from panelprep.scores import SCORES, run_scores

print('registered scores:')
display(SCORES.meta_table()[['orientation', 'applies_to', 'description']])

cfg = Config.from_yaml('configs/default.yaml')
results = run_scores(case.query, case.candidate_list, score_names=cfg.scores,
                     genes=case.query.panel, params=cfg.score_params, seed=cfg.seed,
                     max_obs=cfg.max_obs_per_dataset)
results.pivot_table(index='dataset', columns='score', values='value', dropna=False).reindex(EXPECTED_ORDER).round(4)

### The acceptance check

`shuffled_structure` preserves every gene's marginal exactly while destroying co-expression.
The distributional scores should therefore be close to `sibling`, and the structure scores far
from it. The ratio printed below is the evidence that the two score families measure different
things.

In [ ]:
piv = results.pivot_table(index='dataset', columns='score', values='value', dropna=False)
for s in ['avg_kl', 'avg_js', 'corr_frobenius', 'corr_cosine']:
    a, b = piv.loc['sibling', s], piv.loc['shuffled_structure', s]
    print(f'{s:16s} sibling={a:8.4f}  shuffled={b:8.4f}  ratio={b/a if a else float("nan"):6.2f}')

## The comparison table

Red = strong agreement, blue = weak. Raw values are printed in each cell; the colour is the agreement-normalised rank, because the raw values run in opposite directions (KL and Frobenius are divergences, ARI and cosine are agreements).

In [ ]:
from panelprep.viz import score_heatmap

os.makedirs('outputs/synthetic/figures', exist_ok=True)
fig = score_heatmap(results, out_path='outputs/synthetic/figures/score_heatmap.png',
                    title='Synthetic: panel-gene agreement (expected order: ' + ' > '.join(EXPECTED_ORDER) + ')')
fig

## Adding a score

The point of the registry: define a function, decorate it, add its name to the config. The
heatmap and the markdown report pick it up with no further edits. Run this cell and re-render the
table above to see the new column appear.

In [ ]:
from panelprep.schemas import ScoreResult
from panelprep.scores import SCORES, prepare_panel

@SCORES.register('mean_abs_detection_gap', orientation='lower_is_better',
                 applies_to=('single_cell', 'spatial'), needs_coords=False, units_sensitive=False,
                 description='mean |detection rate difference| over panel genes')
def mean_abs_detection_gap(query, candidate, *, genes, rng, max_obs=None, **_):
    qp, cp, used, missing = prepare_panel(query, candidate, genes, max_obs=max_obs, rng=rng)
    if not used:
        return ScoreResult('mean_abs_detection_gap', candidate.name, None, 'lower_is_better',
                           status='insufficient_genes', n_genes_requested=len(genes))
    gap = float(np.mean(np.abs((qp > 0).mean().to_numpy() - (cp > 0).mean().to_numpy())))
    return ScoreResult('mean_abs_detection_gap', candidate.name, gap, 'lower_is_better',
                       n_genes_used=len(used), n_genes_requested=len(genes),
                       genes_missing=tuple(missing))

extended = run_scores(case.query, case.candidate_list,
                      score_names=cfg.scores + ['mean_abs_detection_gap'],
                      genes=case.query.panel, seed=0, max_obs=cfg.max_obs_per_dataset)
score_heatmap(extended, out_path='outputs/synthetic/figures/score_heatmap_extended.png',
              title='Synthetic: with one extra score added, no other code changed')

## Stages 2-4 and the report

Gene embeddings (separate + joint autoencoders), annotations, and the markdown comparison report.

In [ ]:
from panelprep.pipeline import run_pipeline

out = run_pipeline(case.query, cfg.replace(max_genes=200, embedding_dim=16),
                   candidates=case.candidates, do_annotations=False)
for k, v in (out.get('embeddings') or {}).items():
    print(f'{k:34s} {v.category:14s} {v.matrix.shape}  nan_genes={int(v.matrix.isna().all(axis=1).sum())}')
print('report:', out.get('report_path'))

In [ ]:
from panelprep.viz import embedding_panel, embedding_quality

embs = out['embeddings']
if embs:
    display(pd.DataFrame([{'embedding': k, 'category': v.category, 'shape': str(v.matrix.shape)}
                          for k, v in embs.items()]))
    embedding_panel(embs, None, out_path='outputs/synthetic/figures/embedding_panel.png')